# Valio Aimo – Buffer Margin & Confidence Score Prediction

This notebook builds a unified dataset from:

- `valio_aimo_sales_and_deliveries_junction_2025.csv`
- `valio_aimo_replacement_orders_junction_2025.csv`
- `valio_aimo_purchases_junction_2025.csv`

It then trains models to output:

1. **Shortage probability (confidence score)** per product per day  
2. **Buffer margin prediction** (how many extra units are needed)  
3. **Uncertainty estimates** for buffer (50th, 80th, 95th quantiles)

The design aligns with the Junction 2025 Valio Aimo challenge: proactive, data-driven detection of future stock-outs, to enable voice-first and text-based customer notifications with suggested substitutions.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, r2_score

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [2]:
# Adjust DATA_DIR if your CSVs are in a different folder
DATA_DIR = Path(".")

SALES_FILE = DATA_DIR / "valio_aimo_sales_and_deliveries_junction_2025.csv"
REPL_FILE  = DATA_DIR / "valio_aimo_replacement_orders_junction_2025.csv"
PURCH_FILE = DATA_DIR / "valio_aimo_purchases_junction_2025.csv"

SALES_FILE, REPL_FILE, PURCH_FILE


(PosixPath('valio_aimo_sales_and_deliveries_junction_2025.csv'),
 PosixPath('valio_aimo_replacement_orders_junction_2025.csv'),
 PosixPath('valio_aimo_purchases_junction_2025.csv'))

In [3]:
sales = pd.read_csv(SALES_FILE)
repl  = pd.read_csv(REPL_FILE)
purch = pd.read_csv(PURCH_FILE)

print("Sales shape:", sales.shape)
print("Replacement shape:", repl.shape)
print("Purchases shape:", purch.shape)

sales.head()


Sales shape: (7357509, 18)
Replacement shape: (15069, 18)
Purchases shape: (782783, 11)


,order_number,order_created_date,order_created_time,requested_delivery_date,customer_number,order_row_number,product_code,order_qty,sales_unit,delivery_number,plant,storage_location,delivered_qty,transfer_number,warehouse_number,picking_confirmed_date,picking_confirmed_time,picking_picked_qty
0,10000000,2024-09-01,336,2024-09-02,33258,10,409510,5.0,ST,20000000.0,30588.0,2012.0,5.0,30000212.0,3001.0,2024-09-01,203837.0,5.0
1,10000000,2024-09-01,336,2024-09-02,33258,40,410914,12.0,ST,20000000.0,30588.0,2012.0,12.0,30000212.0,3001.0,2024-09-01,203734.0,12.0
2,10000000,2024-09-01,336,2024-09-02,33258,50,406587,4.0,ST,20000000.0,30588.0,2012.0,4.0,30000211.0,3001.0,2024-09-01,204149.0,4.0
3,10000000,2024-09-01,336,2024-09-02,33258,60,406588,4.0,ST,20000000.0,30588.0,2012.0,4.0,30000211.0,3001.0,2024-09-01,204124.0,4.0
4,10000000,2024-09-01,336,2024-09-02,33258,70,401369,8.0,BOT,20000000.0,30588.0,2012.0,8.0,30000211.0,3001.0,2024-09-01,205255.0,8.0


In [4]:
def parse_date(df, col):
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    return df

# Sales & Replacement date parsing
for df in (sales, repl):
    for col in ["order_created_date", "requested_delivery_date", "picking_confirmed_date"]:
        parse_date(df, col)

# Purchases date parsing
for col in ["po_created_date", "requested_delivery_date"]:
    parse_date(purch, col)

sales.dtypes.head(20)


order_number                        int64
order_created_date         datetime64[ns]
order_created_time                  int64
requested_delivery_date    datetime64[ns]
customer_number                     int64
order_row_number                    int64
product_code                        int64
order_qty                         float64
sales_unit                         object
delivery_number                   float64
plant                             float64
storage_location                  float64
delivered_qty                     float64
transfer_number                   float64
warehouse_number                  float64
picking_confirmed_date     datetime64[ns]
picking_confirmed_time            float64
picking_picked_qty                float64
dtype: object

In [5]:
# Use requested_delivery_date as the main date for demand
sales["date"] = sales["requested_delivery_date"]

sales_agg = (
    sales.groupby(["product_code", "date"], as_index=False)
    .agg(
        total_order_qty=("order_qty", "sum"),
        total_delivered_qty=("delivered_qty", "sum"),
        total_picked_qty=("picking_picked_qty", "sum"),
        n_orders=("order_number", "nunique"),
        n_deliveries=("delivery_number", "nunique"),
    )
)

sales_agg["shortage_amount"] = (
    sales_agg["total_order_qty"] - sales_agg["total_delivered_qty"]
).clip(lower=0)

sales_agg["shortage_flag"] = (sales_agg["shortage_amount"] > 0).astype(int)

sales_agg.head()


,product_code,date,total_order_qty,total_delivered_qty,total_picked_qty,n_orders,n_deliveries,shortage_amount,shortage_flag
0,400001,2025-01-16,1.0,1.0,1.0,1,1,0.0,0
1,400001,2025-03-03,1.0,1.0,1.0,1,1,0.0,0
2,400001,2025-04-02,1.0,1.0,1.0,1,1,0.0,0
3,400001,2025-10-01,1.0,1.0,1.0,1,1,0.0,0
4,400002,2024-09-04,12.0,4.0,4.0,2,1,8.0,1


In [6]:
# Replacement file has the same schema as sales
repl["date"] = repl["requested_delivery_date"]

repl_agg = (
    repl.groupby(["product_code", "date"], as_index=False)
    .agg(
        replacement_order_qty=("order_qty", "sum"),
        replacement_delivered_qty=("delivered_qty", "sum"),
        replacement_picked_qty=("picking_picked_qty", "sum"),
        n_repl_orders=("order_number", "nunique"),
    )
)

repl_agg["replacement_qty"] = (
    repl_agg["replacement_order_qty"] - repl_agg["replacement_delivered_qty"]
).clip(lower=0)

repl_agg["replacement_flag"] = (repl_agg["replacement_qty"] > 0).astype(int)

repl_agg.head()


,product_code,date,replacement_order_qty,replacement_delivered_qty,replacement_picked_qty,n_repl_orders,replacement_qty,replacement_flag
0,400006,2025-06-19,1.0,1.0,1.0,1,0.0,0
1,400006,2025-07-01,1.0,1.0,1.0,1,0.0,0
2,400006,2025-07-14,10.0,10.0,5.0,1,0.0,0
3,400007,2024-12-31,10.0,10.0,10.0,1,0.0,0
4,400008,2025-03-21,2.0,0.0,0.0,1,2.0,1


In [7]:
# Use requested_delivery_date in purchases as supply date
purch["date"] = purch["requested_delivery_date"]

purch_agg = (
    purch.groupby(["product_code", "date"], as_index=False)
    .agg(
        total_po_ordered_qty=("ordered_qty", "sum"),
        total_received_qty=("received_qty", "sum"),
        n_po_rows=("order_number", "nunique"),
    )
)

purch_agg["supply_gap"] = (
    purch_agg["total_po_ordered_qty"] - purch_agg["total_received_qty"]
)

purch_agg.head()


,product_code,date,total_po_ordered_qty,total_received_qty,n_po_rows,supply_gap
0,400001,2025-01-14,1.0,1.0,1,0.0
1,400001,2025-02-27,1.0,1.0,1,0.0
2,400001,2025-04-01,1.0,1.0,1,0.0
3,400001,2025-10-01,1.0,1.0,1,0.0
4,400002,2024-09-04,10.0,10.0,1,0.0


In [8]:
full = sales_agg.merge(
    repl_agg, on=["product_code", "date"], how="outer"
).merge(
    purch_agg, on=["product_code", "date"], how="outer"
)

full.sort_values(["product_code", "date"], inplace=True)
full.reset_index(drop=True, inplace=True)

full.head()


,product_code,date,total_order_qty,total_delivered_qty,total_picked_qty,n_orders,n_deliveries,shortage_amount,shortage_flag,replacement_order_qty,replacement_delivered_qty,replacement_picked_qty,n_repl_orders,replacement_qty,replacement_flag,total_po_ordered_qty,total_received_qty,n_po_rows,supply_gap
0,400001,2025-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0
1,400001,2025-01-16,1.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,400001,2025-02-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0
3,400001,2025-03-03,1.0,1.0,1.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,400001,2025-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,0.0


In [9]:
# Fill numeric NaNs with 0 at this stage
for col in full.select_dtypes(include=np.number).columns:
    full[col] = full[col].fillna(0)

# Ensure date is datetime
full["date"] = pd.to_datetime(full["date"], errors="coerce")

# Combined target: max of sales shortage and replacement qty
full["buffer_target"] = full[["shortage_amount", "replacement_qty"]].max(axis=1)

full[["product_code", "date", "shortage_amount", "replacement_qty", "buffer_target"]].head(10)


,product_code,date,shortage_amount,replacement_qty,buffer_target
0,400001,2025-01-14,0.0,0.0,0.0
1,400001,2025-01-16,0.0,0.0,0.0
2,400001,2025-02-27,0.0,0.0,0.0
3,400001,2025-03-03,0.0,0.0,0.0
4,400001,2025-04-01,0.0,0.0,0.0
5,400001,2025-04-02,0.0,0.0,0.0
6,400001,2025-10-01,0.0,0.0,0.0
7,400002,2024-09-04,8.0,0.0,8.0
8,400002,2024-09-05,0.0,0.0,0.0
9,400002,2024-09-06,0.0,0.0,0.0


In [10]:
full.sort_values(["product_code", "date"], inplace=True)

def add_rolling(df, col, window, prefix):
    rolled = (
        df.groupby("product_code")[col]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).sum())
    )
    df[f"{prefix}_roll{window}"] = rolled.fillna(0)
    return df

# Demand rolling
full = add_rolling(full, "total_order_qty", 7,  "demand")
full = add_rolling(full, "total_order_qty", 30, "demand")

# Supply rolling
full = add_rolling(full, "total_received_qty", 7,  "supply")
full = add_rolling(full, "total_received_qty", 30, "supply")

# Shortage rolling
full = add_rolling(full, "shortage_amount", 7,  "shortage")
full = add_rolling(full, "shortage_amount", 30, "shortage")

full.head()


,product_code,date,total_order_qty,total_delivered_qty,total_picked_qty,n_orders,n_deliveries,shortage_amount,shortage_flag,replacement_order_qty,replacement_delivered_qty,replacement_picked_qty,n_repl_orders,replacement_qty,replacement_flag,total_po_ordered_qty,total_received_qty,n_po_rows,supply_gap,buffer_target,demand_roll7,demand_roll30,supply_roll7,supply_roll30,shortage_roll7,shortage_roll30
0,400001,2025-01-14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,400001,2025-01-16,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,400001,2025-02-27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0
3,400001,2025-03-03,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,0.0,0.0
4,400001,2025-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,2.0,2.0,2.0,2.0,0.0,0.0


In [11]:
# Drop rows with invalid dates (if any)
full = full[full["date"].notnull()].copy()

full["day_of_week"] = full["date"].dt.dayofweek
full["week_of_year"] = full["date"].dt.isocalendar().week.astype(int)
full["month"] = full["date"].dt.month

# Encode product_code as categorical integer
full["product_code_enc"] = full["product_code"].astype("category").cat.codes


In [12]:
feature_cols = [
    "product_code_enc", "day_of_week", "week_of_year", "month",
    "total_order_qty", "total_delivered_qty", "total_picked_qty",
    "n_orders", "n_deliveries",
    "replacement_order_qty", "replacement_delivered_qty",
    "replacement_picked_qty", "n_repl_orders", "replacement_flag",
    "total_po_ordered_qty", "total_received_qty",
    "n_po_rows", "supply_gap",
    "demand_roll7", "demand_roll30",
    "supply_roll7", "supply_roll30",
    "shortage_roll7", "shortage_roll30",
]

# Keep only existing columns (robustness)
feature_cols = [c for c in feature_cols if c in full.columns]

X = full[feature_cols].copy()
y = full["buffer_target"].copy()

X.shape, y.shape


((1475664, 24), (1475664,))

In [13]:
# Fill any remaining NaNs in X to avoid issues in models
X = X.fillna(0)

# Time-based train/test split (80/20) on date
dates_sorted = full["date"].sort_values().unique()
cutoff_index = int(len(dates_sorted) * 0.8)
cutoff_date = dates_sorted[cutoff_index]

train_mask = full["date"] < cutoff_date
test_mask = ~train_mask

X_train = X[train_mask]
y_train = y[train_mask]
X_test  = X[test_mask]
y_test  = y[test_mask]

print("Cutoff date:", cutoff_date)
print("NaNs in X_train:", X_train.isna().sum().sum())
print("NaNs in X_test:", X_test.isna().sum().sum())
X_train.shape, X_test.shape


Cutoff date: 2025-08-31 00:00:00
NaNs in X_train: 0
NaNs in X_test: 0


((1329611, 24), (146053, 24))

In [14]:
buffer_reg = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
)

buffer_reg.fit(X_train, y_train)

y_pred = buffer_reg.predict(X_test)

print("Buffer regression MAE:", mean_absolute_error(y_test, y_pred))
print("Buffer regression R2:", r2_score(y_test, y_pred))


Buffer regression MAE: 0.8437962446154799
Buffer regression R2: 0.7249859840222252


In [15]:
# Classification target: did we need any buffer at all?
y_class = (y > 0).astype(int)
y_train_c = y_class[train_mask]
y_test_c  = y_class[test_mask]

clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
)

clf.fit(X_train, y_train_c)
shortage_prob = clf.predict_proba(X_test)[:, 1]

print("Example shortage probabilities (confidence scores):")
shortage_prob[:10]


KeyboardInterrupt: 

In [ ]:
quantiles = [0.5, 0.8, 0.95]
quantile_models = {}
quantile_preds = {}

for q in quantiles:
    qmodel = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
    )
    qmodel.fit(X_train, y_train)
    quantile_models[q] = qmodel
    quantile_preds[q] = qmodel.predict(X_test)

{q: quantile_preds[q][:5] for q in quantiles}


In [ ]:
results = full.loc[test_mask, ["product_code", "date"]].copy()

results["shortage_probability"] = shortage_prob
results["buffer_pred_median"]   = quantile_preds[0.5]
results["buffer_pred_80"]       = quantile_preds[0.8]
results["buffer_pred_95"]       = quantile_preds[0.95]

# Operational recommended buffer (80th percentile)
results["recommended_buffer"] = results["buffer_pred_80"]

results.head(20)


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, quantile_preds[0.5], alpha=0.3)
plt.xlabel("True buffer_target")
plt.ylabel("Predicted buffer (median, 50th quantile)")
plt.title("True vs Predicted Buffer (Median Quantile)")
min_val = min(y_test.min(), quantile_preds[0.5].min())
max_val = max(y_test.max(), quantile_preds[0.5].max())
plt.plot([min_val, max_val], [min_val, max_val], "--")
plt.tight_layout()
plt.show()


In [ ]:
# Save results for downstream use (agent, dashboards, etc.)
results.to_csv("valio_buffer_predictions_with_confidence.csv", index=False)
results.head()
